# Mid Fusion Lite - Kaggle Run All
Notebook nay duoc thiet ke de bam Run All la train duoc.

## Ban can gan 2 dataset trong Kaggle Notebook
1. Dataset code cua ban (chua folder `mid_fusion_lite`).
2. Dataset ISIC 2024 chua cac file metadata va HDF5 (`train-metadata.csv`, `test-metadata.csv`, `sample_submission.csv`, `train-image.hdf5`, `test-image.hdf5`).

## Quan trong
Chi can sua duong dan o Cell 2 (phan USER CONFIG).

In [1]:
# =========================
# CELL 2A: DATA & PATHS
# =========================
"""Cấu hình vị trí dữ liệu và môi trường chạy"""

from pathlib import Path

# ===== ENVIRONMENT =====
RUN_MODE = 'auto'  # 'auto' | 'kaggle' | 'local'

# ===== KAGGLE PATHS =====
KAGGLE_CODE_DATASET_ROOT = '/kaggle/input/datasets/trnhintmh03/isic-2024-v19'
KAGGLE_COMP_DATASET_ROOT = '/kaggle/input/competitions/isic-2024-challenge'
KAGGLE_OUTPUT_DIR = '/kaggle/working/mid_fusion_lite_outputs'

# ===== LOCAL PATHS =====
LOCAL_CODE_DATASET_ROOT = r"D:\HK1_2025_2026\SEMINAR_CODE\mid_fusion_lite"
LOCAL_DATA_ROOT = r"D:\HK1_2025_2026\SEMINAR_CODE\data"
LOCAL_OUTPUT_DIR = './local_outputs'

# Auto-detection
def _first_existing(candidates):
    for candidate in candidates:
        p = Path(candidate)
        if p.exists():
            return p
    return None

if RUN_MODE not in {'auto', 'kaggle', 'local'}:
    raise ValueError("RUN_MODE must be one of: auto, kaggle, local")

if RUN_MODE == 'auto':
    is_kaggle = Path('/kaggle/input').exists()
else:
    is_kaggle = RUN_MODE == 'kaggle'

# Resolve code root
if is_kaggle:
    code_root = Path(KAGGLE_CODE_DATASET_ROOT)
else:
    if LOCAL_CODE_DATASET_ROOT is not None and str(LOCAL_CODE_DATASET_ROOT).strip() != '':
        code_root = Path(str(LOCAL_CODE_DATASET_ROOT))
    else:
        code_root = _first_existing(['./mid_fusion_lite', '.', '../mid_fusion_lite'])
        if code_root is None:
            code_root = Path('./mid_fusion_lite')

CODE_DATASET_ROOT = str(code_root)

# Resolve data paths
if is_kaggle:
    comp_root = Path(KAGGLE_COMP_DATASET_ROOT)
    TRAIN_METADATA_PATH = str(comp_root / 'train-metadata.csv')
    TEST_METADATA_PATH = str(comp_root / 'test-metadata.csv')
    SAMPLE_SUBMISSION_PATH = str(comp_root / 'sample_submission.csv')
    TRAIN_HDF5_PATH = str(comp_root / 'train-image.hdf5')
    TEST_HDF5_PATH = str(comp_root / 'test-image.hdf5')
    OUTPUT_DIR = KAGGLE_OUTPUT_DIR
else:
    if LOCAL_DATA_ROOT is not None and str(LOCAL_DATA_ROOT).strip() != '':
        local_root = Path(str(LOCAL_DATA_ROOT))
        TRAIN_METADATA_PATH = str(local_root / 'train-metadata.csv')
        TEST_METADATA_PATH = str(local_root / 'test-metadata.csv')
        SAMPLE_SUBMISSION_PATH = str(local_root / 'sample_submission.csv')
        TRAIN_HDF5_PATH = str(local_root / 'train-image.hdf5')
        TEST_HDF5_PATH = str(local_root / 'test-image.hdf5')
    else:
        train_meta = _first_existing(['./data/train-metadata.csv', '../data/train-metadata.csv'])
        test_meta = _first_existing(['./data/test-metadata.csv', '../data/test-metadata.csv'])
        sample_sub = _first_existing(['./data/sample_submission.csv', '../data/sample_submission.csv'])
        train_h5 = _first_existing(['./data/train-image.hdf5', '../data/train-image.hdf5'])
        test_h5 = _first_existing(['./data/test-image.hdf5', '../data/test-image.hdf5'])

        TRAIN_METADATA_PATH = str(train_meta if train_meta is not None else Path('./data/train-metadata.csv'))
        TEST_METADATA_PATH = str(test_meta if test_meta is not None else Path('./data/test-metadata.csv'))
        SAMPLE_SUBMISSION_PATH = str(sample_sub if sample_sub is not None else Path('./data/sample_submission.csv'))
        TRAIN_HDF5_PATH = str(train_h5 if train_h5 is not None else Path('./data/train-image.hdf5'))
        TEST_HDF5_PATH = str(test_h5 if test_h5 is not None else Path('./data/test-image.hdf5'))

    OUTPUT_DIR = LOCAL_OUTPUT_DIR

print('✅ Environment:', 'KAGGLE' if is_kaggle else 'LOCAL')
print('✅ CODE_DATASET_ROOT:', CODE_DATASET_ROOT)
print('✅ OUTPUT_DIR:', OUTPUT_DIR)

✅ Environment: KAGGLE
✅ CODE_DATASET_ROOT: /kaggle/input/datasets/trnhintmh03/isic-2024-v19
✅ OUTPUT_DIR: /kaggle/working/mid_fusion_lite_outputs


In [2]:
# =========================
# CELL 2B: MODEL & DATA CONFIG
# =========================
"""Cấu hình kiến trúc mô hình và xử lý dữ liệu"""

# ===== PRESET & BASIC =====
PRESET = 'full_lite'  # 'smoke' (nhanh) | 'quick' (cân bằng) | 'full_lite' (nặng)
FOLDS = None  # None = all | '0' hoặc '0,1,2' để test nhanh

# ===== IMAGE & BACKBONE =====
IMAGE_SIZE_OVERRIDE = 256
BACKBONE_NAME_OVERRIDE = 'efficientnet_b0'
USE_PRETRAINED_BACKBONE = False
BACKBONE_WEIGHTS_PATH = '/kaggle/input/datasets/trnhintmh03/model-backbone/efficientnet_b0_backbone.pth'
TRAIN_IMAGE_BACKBONE_OVERRIDE = True

# ===== TABULAR & FUSION =====
USE_TABM_TABULAR_OVERRIDE = True
FUSION_MODE_OVERRIDE = 'cross_attention'
CROSS_ATTENTION_HEADS_OVERRIDE = 4
CROSS_ATTENTION_DROPOUT_OVERRIDE = 0.1

# ===== TABM ARCHITECTURE =====
TABM_ARCH_TYPE_OVERRIDE = 'tabm'
TABM_K_OVERRIDE = 16
TABM_N_BLOCKS_OVERRIDE = 3
TABM_D_BLOCK_OVERRIDE = 256
TAB_DROPOUT_OVERRIDE = 0.15

# ===== QUANTILE TRANSFORM =====
USE_QUANTILE_TRANSFORM_OVERRIDE = True
QUANTILE_NOISE_STD_OVERRIDE = 1e-5
# --- Post-processing ---                                                                                                                     
USE_PATIENT_POST_PROCESSING_OVERRIDE = True

print('✅ Preset:', PRESET)
print('✅ Fusion Mode:', FUSION_MODE_OVERRIDE)

✅ Preset: full_lite
✅ Fusion Mode: cross_attention


In [3]:
# =========================
# CELL 2C: TRAINING & FEATURES
# =========================
"""Cấu hình quá trình training và feature engineering"""

# ===== OPTIMIZATION =====
EPOCHS_OVERRIDE = 50
LR_OVERRIDE = 3e-4
WEIGHT_DECAY_OVERRIDE = 2e-4
PATIENCE_OVERRIDE = 10

# ===== BATCH & DATA =====
TRAIN_BATCH_SIZE_OVERRIDE = 32
VALID_BATCH_SIZE_OVERRIDE = 32
NEGATIVE_RATIO_OVERRIDE = 5
NUM_WORKERS_OVERRIDE = None  # Auto: 4 (Kaggle) | 0 (Local)

# Auto worker policy
if NUM_WORKERS_OVERRIDE is None:
    NUM_WORKERS_OVERRIDE = 4 if is_kaggle else 0

# ===== FEATURE ENGINEERING =====
USE_FEATURE_ENGINEERING_OVERRIDE = True

# --- Context features (recommended) ---
USE_UGLY_DUCKLING_FEATURES_OVERRIDE = True

# --- Heavy features (only on strong hardware) ---

# --- Null importance filtering ---
USE_NULL_IMPORTANCE_FILTER_OVERRIDE = False
NULL_IMPORTANCE_REPEATS_OVERRIDE = 8
NULL_IMPORTANCE_QUANTILE_OVERRIDE = 0.95
NULL_IMPORTANCE_MIN_KEPT_OVERRIDE = 64

# ===== LOGGING & DEBUG =====
SHOW_PREDICT_PROGRESS_OVERRIDE = True
PREDICT_LOG_INTERVAL_OVERRIDE = 20
MAX_TRAIN_STEPS_OVERRIDE = None
MAX_VALID_STEPS_OVERRIDE = None


# ===== STAGED TRAINING =====
STAGED_TRAINING_OVERRIDE = False  # Set to True to enable 3-stage progressive training
STAGE1_EPOCHS_OVERRIDE = None
STAGE2_EPOCHS_OVERRIDE = None
STAGE3_FREEZE_EPOCHS_OVERRIDE = None
STAGE3_UNFREEZE_EPOCHS_OVERRIDE = None

print('✅ Epochs:', EPOCHS_OVERRIDE)
print('✅ Learning Rate:', LR_OVERRIDE)
print('✅ Feature Engineering:', USE_FEATURE_ENGINEERING_OVERRIDE)
print('✅ NUM_WORKERS:', NUM_WORKERS_OVERRIDE)print('✅ Staged Training:', STAGED_TRAINING_OVERRIDE)


✅ Epochs: 50
✅ Learning Rate: 0.0003
✅ Feature Engineering: True
✅ NUM_WORKERS: 4


In [4]:
from pathlib import Path
import os

if Path('/kaggle/input').exists():
    print('Kaggle input datasets:')
    print(os.listdir('/kaggle/input'))
else:
    print('Kaggle input not found. Running in local mode.')

paths_to_check = {
    'CODE_DATASET_ROOT': CODE_DATASET_ROOT,
    'TRAIN_METADATA_PATH': TRAIN_METADATA_PATH,
    'TEST_METADATA_PATH': TEST_METADATA_PATH,
    'SAMPLE_SUBMISSION_PATH': SAMPLE_SUBMISSION_PATH,
    'TRAIN_HDF5_PATH': TRAIN_HDF5_PATH,
    'TEST_HDF5_PATH': TEST_HDF5_PATH,
}

missing = []
for key, value in paths_to_check.items():
    exists = Path(value).exists()
    print(f'{key}: {value} | exists={exists}')
    if not exists:
        missing.append(key)

runner_path = Path(CODE_DATASET_ROOT) / 'runner.py'
runner_exists = runner_path.exists()
print(f'RUNNER_PATH: {runner_path} | exists={runner_exists}')
if not runner_exists:
    missing.append('CODE_DATASET_ROOT/runner.py')

if missing:
    raise FileNotFoundError(
        'Thieu duong dan: ' + ', '.join(missing) +
        '. Hay sua lai CELL 2 cho dung ten dataset/path.'
    )

print('Path check OK.')

Kaggle input datasets:
['competitions', 'pm-120288904-at-06-11-2026-16-02-13', 'datasets']
CODE_DATASET_ROOT: /kaggle/input/datasets/trnhintmh03/isic-2024-v19 | exists=True
TRAIN_METADATA_PATH: /kaggle/input/competitions/isic-2024-challenge/train-metadata.csv | exists=True
TEST_METADATA_PATH: /kaggle/input/competitions/isic-2024-challenge/test-metadata.csv | exists=True
SAMPLE_SUBMISSION_PATH: /kaggle/input/competitions/isic-2024-challenge/sample_submission.csv | exists=True
TRAIN_HDF5_PATH: /kaggle/input/competitions/isic-2024-challenge/train-image.hdf5 | exists=True
TEST_HDF5_PATH: /kaggle/input/competitions/isic-2024-challenge/test-image.hdf5 | exists=True
RUNNER_PATH: /kaggle/input/datasets/trnhintmh03/isic-2024-v19/runner.py | exists=True
Path check OK.


In [5]:
# Install tabm & rtdl_num_embeddings offline/online first
import sys
from pathlib import Path

offline_dir = Path('/kaggle/input/tabm-kaggle-wheels')
if offline_dir.exists():
    print('Installing tabm and rtdl_num_embeddings offline...')
    !pip install /kaggle/input/tabm-kaggle-wheels/tabm-0.0.3-py3-none-any.whl /kaggle/input/tabm-kaggle-wheels/rtdl_num_embeddings-0.0.12-py3-none-any.whl --no-deps -q
else:
    print('Installing tabm and rtdl_num_embeddings online...')
    !pip install -q tabm rtdl_num_embeddings

# Then install other requirements from requirements.txt
!pip -q install -r "{CODE_DATASET_ROOT}/requirements.txt"

In [6]:
import importlib
import sys
from pathlib import Path

if CODE_DATASET_ROOT in sys.path:
    sys.path.remove(CODE_DATASET_ROOT)
sys.path.insert(0, CODE_DATASET_ROOT)

# Reload modules to use latest local files
import config as config_module
import tabular_engineering_02 as tabular_engineering_module
import tabular_preprocessor_04 as tabular_preprocessor_module
import train as train_module
import runner as runner_module

config_module = importlib.reload(config_module)
tabular_engineering_module = importlib.reload(tabular_engineering_module)
tabular_preprocessor_module = importlib.reload(tabular_preprocessor_module)
train_module = importlib.reload(train_module)
runner_module = importlib.reload(runner_module)

print('✅ Using runner from:', Path(runner_module.__file__).resolve())
print('✅ Using config from:', Path(config_module.__file__).resolve())
print('✅ Using tabular_engineering from:', Path(tabular_engineering_module.__file__).resolve())
print('✅ Using tabular_preprocessor from:', Path(tabular_preprocessor_module.__file__).resolve())
print('✅ Using train from:', Path(train_module.__file__).resolve())

from runner import run_with_preset

overrides = {
    'output_dir': OUTPUT_DIR,
    'train_metadata_path': TRAIN_METADATA_PATH,
    'test_metadata_path': TEST_METADATA_PATH,
    'sample_submission_path': SAMPLE_SUBMISSION_PATH,
    'train_hdf5_path': TRAIN_HDF5_PATH,
    'test_hdf5_path': TEST_HDF5_PATH,
    'use_pretrained_backbone': bool(USE_PRETRAINED_BACKBONE),
}

if BACKBONE_WEIGHTS_PATH is not None and str(BACKBONE_WEIGHTS_PATH).strip() != '':
    overrides['backbone_weights_path'] = str(BACKBONE_WEIGHTS_PATH)

if FOLDS is not None and str(FOLDS).strip() != '':
    overrides['fold_indices'] = [int(x.strip()) for x in str(FOLDS).split(',') if x.strip()]

if EPOCHS_OVERRIDE is not None:
    overrides['epochs'] = int(EPOCHS_OVERRIDE)

if IMAGE_SIZE_OVERRIDE is not None:
    overrides['image_size'] = int(IMAGE_SIZE_OVERRIDE)

if BACKBONE_NAME_OVERRIDE is not None and str(BACKBONE_NAME_OVERRIDE).strip() != '':
    overrides['backbone_name'] = str(BACKBONE_NAME_OVERRIDE).strip()

if TRAIN_IMAGE_BACKBONE_OVERRIDE is not None:
    overrides['train_image_backbone'] = bool(TRAIN_IMAGE_BACKBONE_OVERRIDE)

use_feature_engineering_override = globals().get('USE_FEATURE_ENGINEERING_OVERRIDE', None)
if use_feature_engineering_override is not None:
    overrides['use_feature_engineering'] = bool(use_feature_engineering_override)


if USE_UGLY_DUCKLING_FEATURES_OVERRIDE is not None:
    overrides['use_ugly_duckling_features'] = bool(USE_UGLY_DUCKLING_FEATURES_OVERRIDE)





if USE_NULL_IMPORTANCE_FILTER_OVERRIDE is not None:
    overrides['use_null_importance_filter'] = bool(USE_NULL_IMPORTANCE_FILTER_OVERRIDE)

if NULL_IMPORTANCE_REPEATS_OVERRIDE is not None:
    overrides['null_importance_repeats'] = int(NULL_IMPORTANCE_REPEATS_OVERRIDE)

if NULL_IMPORTANCE_QUANTILE_OVERRIDE is not None:
    overrides['null_importance_quantile'] = float(NULL_IMPORTANCE_QUANTILE_OVERRIDE)

if NULL_IMPORTANCE_MIN_KEPT_OVERRIDE is not None:
    overrides['null_importance_min_kept'] = int(NULL_IMPORTANCE_MIN_KEPT_OVERRIDE)

if USE_TABM_TABULAR_OVERRIDE is not None:
    overrides['use_tabm_tabular'] = bool(USE_TABM_TABULAR_OVERRIDE)

if FUSION_MODE_OVERRIDE is not None and str(FUSION_MODE_OVERRIDE).strip() != '':
    overrides['fusion_mode'] = str(FUSION_MODE_OVERRIDE).strip()

if CROSS_ATTENTION_HEADS_OVERRIDE is not None:
    overrides['cross_attention_heads'] = int(CROSS_ATTENTION_HEADS_OVERRIDE)

if CROSS_ATTENTION_DROPOUT_OVERRIDE is not None:
    overrides['cross_attention_dropout'] = float(CROSS_ATTENTION_DROPOUT_OVERRIDE)

if USE_QUANTILE_TRANSFORM_OVERRIDE is not None:
    overrides['use_quantile_transform'] = bool(USE_QUANTILE_TRANSFORM_OVERRIDE)

if QUANTILE_NOISE_STD_OVERRIDE is not None:
    overrides['quantile_noise_std'] = float(QUANTILE_NOISE_STD_OVERRIDE)

if TABM_ARCH_TYPE_OVERRIDE is not None and str(TABM_ARCH_TYPE_OVERRIDE).strip() != '':
    overrides['tabm_arch_type'] = str(TABM_ARCH_TYPE_OVERRIDE).strip()

if TABM_K_OVERRIDE is not None:
    overrides['tabm_k'] = int(TABM_K_OVERRIDE)

if TABM_N_BLOCKS_OVERRIDE is not None:
    overrides['tabm_n_blocks'] = int(TABM_N_BLOCKS_OVERRIDE)

if TABM_D_BLOCK_OVERRIDE is not None:
    overrides['tabm_d_block'] = int(TABM_D_BLOCK_OVERRIDE)

if TAB_DROPOUT_OVERRIDE is not None:
    overrides['tab_dropout'] = float(TAB_DROPOUT_OVERRIDE)

if LR_OVERRIDE is not None:
    overrides['lr'] = float(LR_OVERRIDE)

if WEIGHT_DECAY_OVERRIDE is not None:
    overrides['weight_decay'] = float(WEIGHT_DECAY_OVERRIDE)

if PATIENCE_OVERRIDE is not None:
    overrides['patience'] = int(PATIENCE_OVERRIDE)

if NEGATIVE_RATIO_OVERRIDE is not None:
    overrides['negative_ratio'] = int(NEGATIVE_RATIO_OVERRIDE)

if NUM_WORKERS_OVERRIDE is not None:
    overrides['num_workers'] = int(NUM_WORKERS_OVERRIDE)

if TRAIN_BATCH_SIZE_OVERRIDE is not None:
    overrides['train_batch_size'] = int(TRAIN_BATCH_SIZE_OVERRIDE)

if VALID_BATCH_SIZE_OVERRIDE is not None:
    overrides['valid_batch_size'] = int(VALID_BATCH_SIZE_OVERRIDE)

if SHOW_PREDICT_PROGRESS_OVERRIDE is not None:
    overrides['show_predict_progress'] = bool(SHOW_PREDICT_PROGRESS_OVERRIDE)

if PREDICT_LOG_INTERVAL_OVERRIDE is not None:
    overrides['predict_log_interval'] = int(PREDICT_LOG_INTERVAL_OVERRIDE)

if MAX_TRAIN_STEPS_OVERRIDE is not None:
    overrides['max_train_steps_per_epoch'] = int(MAX_TRAIN_STEPS_OVERRIDE)

if MAX_VALID_STEPS_OVERRIDE is not None:
    overrides['max_valid_steps'] = int(MAX_VALID_STEPS_OVERRIDE)
    
if USE_PATIENT_POST_PROCESSING_OVERRIDE is not None:                                                                                          
        overrides['use_patient_post_processing'] = bool(USE_PATIENT_POST_PROCESSING_OVERRIDE)


# ===== STAGED TRAINING OVERRIDES =====
staged_training_override = globals().get('STAGED_TRAINING_OVERRIDE', None)
if staged_training_override is not None:
    overrides['staged_training'] = bool(staged_training_override)

stage1_epochs_override = globals().get('STAGE1_EPOCHS_OVERRIDE', None)
if stage1_epochs_override is not None:
    overrides['stage1_epochs'] = int(stage1_epochs_override)

stage2_epochs_override = globals().get('STAGE2_EPOCHS_OVERRIDE', None)
if stage2_epochs_override is not None:
    overrides['stage2_epochs'] = int(stage2_epochs_override)

stage3_freeze_override = globals().get('STAGE3_FREEZE_EPOCHS_OVERRIDE', None)
if stage3_freeze_override is not None:
    overrides['stage3_freeze_epochs'] = int(stage3_freeze_override)

stage3_unfreeze_override = globals().get('STAGE3_UNFREEZE_EPOCHS_OVERRIDE', None)
if stage3_unfreeze_override is not None:
    overrides['stage3_unfreeze_epochs'] = int(stage3_unfreeze_override)

print('\n📋 Running with preset:', PRESET)
print('📋 Overrides:', len(overrides), 'parameters')

result = run_with_preset(PRESET, **overrides)

✅ Using runner from: /kaggle/input/datasets/trnhintmh03/isic-2024-v19/runner.py
✅ Using config from: /kaggle/input/datasets/trnhintmh03/isic-2024-v19/config.py
✅ Using tabular_engineering from: /kaggle/input/datasets/trnhintmh03/isic-2024-v19/tabular_engineering_02.py
✅ Using tabular_preprocessor from: /kaggle/input/datasets/trnhintmh03/isic-2024-v19/tabular_preprocessor_04.py
✅ Using train from: /kaggle/input/datasets/trnhintmh03/isic-2024-v19/train.py

📋 Running with preset: full_lite
📋 Overrides: 39 parameters
Running preset: full_lite
Device: CUDA
Output directory: /kaggle/working/mid_fusion_lite_outputs
Numeric features: 750
Categorical features: 6
Selected folds: [0, 1, 2, 3, 4]
Fold 1/5


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


Loaded local backbone weights: /kaggle/input/datasets/trnhintmh03/model-backbone/efficientnet_b0_backbone.pth | missing=0 | unexpected=0
Epoch 01 | train_loss=3.71461 | valid_loss=1.42294 | valid_auroc=0.87373 | valid_pauc=0.11108
Epoch 02 | train_loss=2.53172 | valid_loss=1.44498 | valid_auroc=0.94161 | valid_pauc=0.15932
Epoch 03 | train_loss=1.94840 | valid_loss=0.97866 | valid_auroc=0.93919 | valid_pauc=0.16237
Epoch 04 | train_loss=1.81580 | valid_loss=0.74870 | valid_auroc=0.93184 | valid_pauc=0.15441
Epoch 05 | train_loss=1.71525 | valid_loss=1.00363 | valid_auroc=0.93747 | valid_pauc=0.15801
Epoch 06 | train_loss=1.56824 | valid_loss=0.50000 | valid_auroc=0.94817 | valid_pauc=0.16244
Epoch 07 | train_loss=1.36885 | valid_loss=0.66276 | valid_auroc=0.94585 | valid_pauc=0.16098
Epoch 08 | train_loss=1.46965 | valid_loss=0.68935 | valid_auroc=0.94934 | valid_pauc=0.16439
Epoch 09 | train_loss=1.23837 | valid_loss=0.31792 | valid_auroc=0.94650 | valid_pauc=0.16134
Epoch 10 | train_

In [7]:
import json
import pandas as pd
import shutil
from pathlib import Path

print('CV pAUC:', result['cv_pauc'])
print('OOF file:', result['oof_path'])
print('Submission file:', result['submission_path'])
print('Metrics file:', result['metrics_path'])

metrics = json.loads(Path(result['metrics_path']).read_text())
print('Fold metrics:')
for row in metrics.get('fold_metrics', []):
    print(row)

# Luu them mot file submission.csv o /kaggle/working de gui len competition nhanh
shutil.copy(result['submission_path'], '/kaggle/working/submission.csv')
print('Copied to /kaggle/working/submission.csv')

display(pd.read_csv('/kaggle/working/submission.csv').head())

CV pAUC: 0.16560825957605008
OOF file: /kaggle/working/mid_fusion_lite_outputs/oof_predictions.csv
Submission file: /kaggle/working/mid_fusion_lite_outputs/submission_mid_fusion_lite.csv
Metrics file: /kaggle/working/mid_fusion_lite_outputs/metrics.json
Fold metrics:
{'fold': 0, 'best_valid_pauc': 0.16438813483277095, 'fold_valid_pauc': 0.16438813483277095, 'n_train': 329895, 'n_valid': 71164, 'epochs_ran': 19}
{'fold': 1, 'best_valid_pauc': 0.17475709585161975, 'fold_valid_pauc': 0.17475709585161975, 'n_train': 313765, 'n_valid': 87294, 'epochs_ran': 25}
{'fold': 2, 'best_valid_pauc': 0.1849011836480419, 'fold_valid_pauc': 0.1849011836480419, 'n_train': 323414, 'n_valid': 77645, 'epochs_ran': 23}
{'fold': 3, 'best_valid_pauc': 0.1765365619208097, 'fold_valid_pauc': 0.1765365619208097, 'n_train': 317698, 'n_valid': 83361, 'epochs_ran': 29}
{'fold': 4, 'best_valid_pauc': 0.18317590018529306, 'fold_valid_pauc': 0.18317590018529306, 'n_train': 319464, 'n_valid': 81595, 'epochs_ran': 22}
C

,isic_id,target
0,ISIC_0015657,0.0
1,ISIC_0015729,0.0
2,ISIC_0015740,0.0
